# Notebook 1 — NIRSpec/BOTS Stage 1 + Stage 2 with `transitspectroscopy`

**Goal:** start from JWST `*_uncal.fits` files and produce a first extracted white-light curve and spectra using the `transitspectroscopy` wrappers.



## 0. Install packages

Run this once if needed. If you already have a JWST environment, you can skip this cell.

In [2]:
# !pip install jwst astroquery astropy matplotlib numpy ray juliet transitspectroscopy
# if you're doing this on the terminal don't forget to also pip install jupyterlab and ipykernel so you can... actually open this...
!free -h

               total        used        free      shared  buff/cache   available
Mem:            14Gi       1.5Gi        13Gi        31Mi       378Mi        13Gi
Swap:          2.0Gi       747Mi       1.3Gi


## 1. Imports and CRDS setup

CRDS is needed by the JWST pipeline to find calibration reference files.

In [3]:
import os
import glob
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
# If that fails (very possible for this package)

# Then the package is simply n
import jwst
import transitspectroscopy as ts
from jwst import datamodels

# CRDS setup: change CRDS_PATH if you want the cache somewhere else.
os.environ["CRDS_PATH"] = os.path.expanduser("~/crds_cache")
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

print("jwst version:", jwst.__version__)
print("transitspectroscopy version:", getattr(ts, "__version__", "unknown"))
print("CRDS_PATH:", os.environ["CRDS_PATH"])

jwst version: 1.17.1
transitspectroscopy version: 0.3.12
CRDS_PATH: /home/peng/crds_cache


## 2. User settings

Edit these paths and the detector.

For NIRSpec/BOTS, common detectors are `nrs1` and `nrs2`.

In [4]:
# Input folder containing *_uncal.fits files
data_dir = Path("/media/peng/KINGSTON/mastDownload/JWST/TOI-1130b/NIRSpec/uncal/nrs1/jw03385001001_04102_00001-seg001_nrs1")

# Output folders
stage1_dir = Path("/home/peng/S26_outputs/TOI-1130b/TOI-1130b_S1")
stage2_dir = Path("/home/peng/S26_outputs/TOI-1130b/TOI-1130b_S2")

stage1_dir.mkdir(exist_ok=True, parents=True)
stage2_dir.mkdir(exist_ok=True, parents=True)

# Choose detector
detector = "nrs1"
# detector = "nrs2"

# File pattern. Keep it broad at first, then make it more specific if needed.
uncal_pattern = f"**/*_{detector}_uncal.fits"

# Name used for saved products
target_label = "TOI-1130b"
output_label = f"{target_label}_{detector}"
print('ok')

ok


## 3. Find the `uncal` files

Stage 1 starts from `*_uncal.fits` files.

In [5]:
uncal_files = sorted(data_dir.glob(uncal_pattern))
uncal_files = [str(f) for f in uncal_files]

print(f"Found {len(uncal_files)} uncal files for {detector}")
for f in uncal_files[:10]:
    print(f)

if len(uncal_files) == 0:
    raise FileNotFoundError(
        "No uncal files found. Check data_dir, detector, and uncal_pattern."
    )

Found 1 uncal files for nrs1
/media/peng/KINGSTON/mastDownload/JWST/TOI-1130b/NIRSpec/uncal/nrs1/jw03385001001_04102_00001-seg001_nrs1/jw03385001001_04102_00001-seg001_nrs1_uncal.fits


## 4. Quick check of the first file

This verifies that the files can be opened and shows basic exposure information.

In [6]:
with datamodels.RampModel(uncal_files[0]) as model:
    print("Instrument:", model.meta.instrument.name)
    print("Detector:", model.meta.instrument.detector)
    print("Exposure type:", model.meta.exposure.type)
    print("NINTS:", model.meta.exposure.nints)
    print("NGROUPS:", model.meta.exposure.ngroups)
    print("Data shape:", model.data.shape)

Instrument: NIRSPEC
Detector: NRS1
Exposure type: NRS_BRIGHTOBJ
NINTS: 850
NGROUPS: 8
Data shape: (850, 8, 32, 2048)


## 5. Run Stage 1

This runs the detector-level processing and produces ramp-fitted products per integration.

For a first run, keep `maximum_cores="1"`. Increase it only after the notebook works.

In [ ]:
stage1_output ={}
stage1_output = ts.stage1(
    uncal_files[0],
    background_model=None,
    outputfolder=str(stage1_dir),
    use_tso_jump=True,
    ommit_pixeldq=False,
    maximum_cores="1",
)

print("Stage 1 complete")
print(stage1_output.keys())


	 	 >> Warning: model.meta.dither.dither_points gave  None

	 	 >> Setting manually to 1.
	 >> dqinit step products found, loading them...


2026-05-25 14:35:33,342 - CRDS - INFO -  Calibration SW Found: jwst 1.17.1 (/home/peng/anaconda3/envs/snvenv/lib/python3.10/site-packages/jwst-1.17.1.dist-info)
2026-05-25 14:35:33,792 - stpipe.SaturationStep - INFO - SaturationStep instance created.
2026-05-25 14:35:33,955 - stpipe.SaturationStep - INFO - Step SaturationStep running with args (<RampModel(850, 8, 32, 2048) from jw03385001001_04102_00001-seg001_nrs1_dqinitstep.fits>,).
2026-05-25 14:35:33,957 - stpipe.SaturationStep - INFO - Step SaturationStep parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/peng/S26_outputs/TOI-1130b/TOI-1130b_S1/pipeline_outputs
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  n_pix_grow_sat: 1
  use_readpatt: True
2026-05-25 14:35:33,976 - stpipe.SaturationStep - INFO - Using SATURATION reference file /home/peng/crds_cache/references/jwst/nirspe

## 6. Save Stage 1 output

This avoids rerunning Stage 1 every time.

In [ ]:
stage1_pickle = stage1_dir / f"{output_label}_stage1_output.pkl"

with open(stage1_pickle, "wb") as f:
    pickle.dump(stage1_output, f)

print("Saved:", stage1_pickle)

## 7. Run Stage 2 extraction

This extracts simple spectra and a white-light curve.

Start with `aperture_radius=3`. Later you can test other values.

In [ ]:
stage2_output = ts.stage2(
    stage1_output,
    aperture_radius=3,
    outputfolder=str(stage2_dir),
)

print("Stage 2 complete")
print(stage2_output.keys())

## 8. Sort everything by time

This is useful before plotting or fitting the light curve.

In [ ]:
times = np.asarray(stage2_output["spectra"]["times"])
sort_idx = np.argsort(times)

stage2_output["spectra"]["times"] = times[sort_idx]
stage2_output["whitelight"] = np.asarray(stage2_output["whitelight"])[sort_idx]
stage2_output["whitelight_err"] = np.asarray(stage2_output["whitelight_err"])[sort_idx]

for key in ["corrected", "corrected_err"]:
    if key in stage2_output["spectra"]:
        stage2_output["spectra"][key] = np.asarray(stage2_output["spectra"][key])[sort_idx, :]

print("Sorted by time")

## 9. Plot the white-light curve

This is only a first diagnostic plot.

In [ ]:
t = stage2_output["spectra"]["times"]
flux = stage2_output["whitelight"]
err = stage2_output["whitelight_err"]

plt.figure(figsize=(10, 4))
plt.errorbar(t, flux, yerr=err, fmt=".", alpha=0.7)
plt.xlabel("Time [BJD_TDB]")
plt.ylabel("White-light flux")
plt.title(f"NIRSpec {detector.upper()} white-light curve")
plt.tight_layout()
plt.show()

## 10. Plot a few spectra

This checks that the extraction looks reasonable.

In [ ]:
spectra = stage2_output["spectra"]

# Try common wavelength keys used by extraction dictionaries.
wavelength_key = None
for key in ["wavelength", "wavelengths", "wave"]:
    if key in spectra:
        wavelength_key = key
        break

if wavelength_key is None:
    print("Available spectra keys:", spectra.keys())
    print("No wavelength key found automatically.")
else:
    wave = np.asarray(spectra[wavelength_key])
    spec = np.asarray(spectra["corrected"])

    plt.figure(figsize=(10, 4))
    for i in np.linspace(0, spec.shape[0] - 1, 5, dtype=int):
        plt.plot(wave, spec[i], alpha=0.7, label=f"int {i}")
    plt.xlabel("Wavelength [micron]")
    plt.ylabel("Extracted flux")
    plt.title(f"Example extracted spectra — {detector.upper()}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 11. Save Stage 2 output

This file is the starting point for the next notebook: light-curve fitting and spectral binning.

In [ ]:
stage2_pickle = stage2_dir / f"{output_label}_stage2_spectra.pkl"

with open(stage2_pickle, "wb") as f:
    pickle.dump(stage2_output, f)

print("Saved:", stage2_pickle)

## 12. Minimal checklist 

Before moving to light-curve fitting, check:

1. Did Stage 1 finish without errors?
2. Did Stage 2 finish without errors?
3. Does the white-light curve show a transit/eclipse-like shape?
4. Do the extracted spectra look smooth enough for a first reduction?
5. Are the saved pickle files in the expected folders?

Next notebook: clean bad points, bin spectroscopic channels, and fit the white-light curve.